# Projet-5GENE:
- Geoffrey RIVIERE
- Clara DESTREBECQ
- Yann LAURENT

## Partie 1 : Le format MIDI

### 1.1 Qu'est-ce que le MIDI ? 
Le MIDI, ou **Musical Instrument Digital Interface**, est un protocole de communication créé dans les années 80 pour permettre aux instruments de musique électroniques et aux ordinateurs de communiquer entre eux.

Il transmet des instructions musicales : les notes jouées et leur intensité. Un fichier MIDI ne contient pas de son, mais une partition numérique pour permettre à un ordinateur ou un synthétiseur de l'interpréter.

### 1.2 Format des fichiers
Le format de fichier SMF, **standard MIDI File**, a été défini en 1987, pour stocker les commandes MIDI sur disquette en y ajoutant des informations temporelles avec comme extension **.mid** ou **.midi**.

Il existe 3 formats SMF différents de fichiers MIDI :
- 0 : une seule piste contenant tous les messages des 16 canaux.
- 1 : plusieurs pistes, jouées simultanément.
- 2 : plusieurs pistes, mais jouées séquentiellement (rarement utilisé).

Bien que le type **1** soit le format le plus courant et le plus pratique, certains appareils ne reconnaissent que le format 0.

Les logiciels MIDI ont généralement un format de fichier propriétaire pour gérer les séquences MIDI mais sont tous capables d'importer et d'exporter les fichiers **.mid**, ce qui facilite les échanges entre plateformes.

### 1.3 Avantages du format MIDI
- **Léger** : un fichier MIDI ne contient que des instructions, il ne contient que quelques Ko pour plusieurs minutes de musique, contre plusieurs Mo pour un fichier audio style MP3.
- **Standard universel** : la quasi-totalité des logiciels et instruments de musique l'acceptent, ce qui facilite les échanges.
- **Modifiable facilement** : on peut changer le tempo, l'instrument et transposer les notes sans perdre de la qualité.
- **Données symboliques** : les notes sont directement accessibles sous forme de nombres (hauteur, durée, vélocité), ce qui les rend faciles à exploiter en apprentissage automatique.

**Limites** :
- Le rendu sonore dépend de l'appareil ou du logiciel qui va lire le fichier.
- Le MIDI ne contient pas la voix ni le timbre réel d'un instrument enregistré.

### 1.4 Caractéristiques d'une note
- **Hauteur** : nombre entre 0 et 127
- **Vélocité** : un nombre entre 0 et 127, qui correspond à la force de frappe
- **Durée** : elle n'est pas écrite directement, elle se calcule avec l'écart de temps entre le **note_on** et le **note_off**

Le début et la fin d'une note sont indiqués par des messages, les plus importants étant :
- **note_on** : Début d'une note (hauteur + vélocité)
- **note_off** : Fin d'une note

### Sources
- Wikipédia : **https://fr.wikipedia.org/wiki/Musical_Instrument_Digital_Interface**
- Amped Studio: **https://ampedstudio.com/fr/blog/what-is-midi/**

## Partie 2 : Choix d'une librairie Python

### 2.1 Comparaison des Librairies.
- **Mido** :
    - **Avantages** :
        - Accès direct aux ports, aux messages et aux fichiers MIDI ("MIDI 1.0")
        - Message représenté par un objet Python
        - Utile pour lire des messages un par un
    - **Inconvénients** :
        - Ne donne pas de notes complètes, il faut les reconstruire à partir des messages
- **pretty_midi** :
    - **Avantages** :
        - Facile à gérer (**modifier et à analyser**)
        - Chaque note possède une hauteur ("pitch"), une vélocité ("velocity"), un début ("start") et une fin ("end")
    - **Inconvénients** :
        - Accés moins direct au messages que mido

**Choix** : Nous utiliserons donc **pretty_midi**, car chaque note contient déjà les informations nécessaires pour obtenir les notes complètes et leurs durées.

### 2.2 Fonctionnement de pretty_midi.

**pretty_midi** organise un fichier MIDI en trois niveaux :

- **PrettyMIDI** : représente le fichier MIDI complet.
    - "instruments" : liste d'objets Instrument
- **Instrument** : Représente un instrument MIDI
    - "notes" : liste d'objets note
    - "program" : type d'instrument
    - "is_drum" : booléan indiquant si c'est une piste de percussions
- **Notes** : représente une note jouée.
    - "pitch" : hauteur de la note (0-127)
    - "velocity" : vélocité de la note (en secondes)
    - "start" : début de la note (en secondes)
    - "end" : fin de la note (0-127)

### Sources
- Mido : **https://mido.readthedocs.io/en/stable/**
- pretty_midi: **https://craffel.github.io/pretty-midi/**

## Partie 3 : Choix du compositeur

### 3.1 Lire le fichier

In [1]:
import pandas as pd

df = pd.read_csv("../data/maestro-v3.0.0/maestro-v3.0.0.csv")
df.head()

,canonical_composer,canonical_title,split,year,midi_filename,audio_filename,duration
0,Alban Berg,Sonata Op. 1,train,2018,2018/MIDI-Unprocessed_Chamber3_MID--AUDIO_10_R...,2018/MIDI-Unprocessed_Chamber3_MID--AUDIO_10_R...,698.661160
1,Alban Berg,Sonata Op. 1,train,2008,2008/MIDI-Unprocessed_03_R2_2008_01-03_ORIG_MI...,2008/MIDI-Unprocessed_03_R2_2008_01-03_ORIG_MI...,759.518471
2,Alban Berg,Sonata Op. 1,train,2017,2017/MIDI-Unprocessed_066_PIANO066_MID--AUDIO-...,2017/MIDI-Unprocessed_066_PIANO066_MID--AUDIO-...,464.649433
3,Alexander Scriabin,"24 Preludes Op. 11, No. 13-24",train,2004,2004/MIDI-Unprocessed_XP_21_R1_2004_01_ORIG_MI...,2004/MIDI-Unprocessed_XP_21_R1_2004_01_ORIG_MI...,872.640588
4,Alexander Scriabin,"3 Etudes, Op. 65",validation,2006,2006/MIDI-Unprocessed_17_R1_2006_01-06_ORIG_MI...,2006/MIDI-Unprocessed_17_R1_2006_01-06_ORIG_MI...,397.857508


### 3.2 Calculer la durée et le nombre de morceaux par compositeur.

In [8]:
durees = (
    df.groupby("canonical_composer")["duration"].agg(nb_morceaux="count", duree_secondes="sum").sort_values("duree_secondes", ascending=False)
)
durees["duree_heures"] = (durees["duree_secondes"] / 3600).round(1)

durees.head(10)

,nb_morceaux,duree_secondes,duree_heures
canonical_composer,,,
Franz Schubert,186,128958.701199,35.8
Ludwig van Beethoven,146,96922.682897,26.9
Frédéric Chopin,201,94389.006934,26.2
Franz Liszt,131,72958.738892,20.3
Robert Schumann,49,56596.114873,15.7
Johann Sebastian Bach,145,49559.997408,13.8
Johannes Brahms,26,29028.633682,8.1
Sergei Rachmaninoff,59,23985.896308,6.7
Claude Debussy,45,21507.282772,6.0


### 3.3 Choix du compositeur

Nous allons donc choisir **Franz Schubert**, il s'agit du compositeur qui a la plus longue durée totale dans le dataset, environ 36 heures, réparties dans 186 morceaux.